In [ ]:
# ============================================================================
# CONFOUND ANALYSIS — Standalone cell
# Tests whether disagreement → FP survives controlling for chunk length,
# article length, and TF-IDF overlap. Cluster-robust logistic regressions
# on the 565 true negatives.
#
# Self-contained: loads the benchmark CSV + OOF probas from disk,
# uses prefixed variable names to avoid collisions with other cells.
# ============================================================================

import os, re, sys, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# 1) Configuration (edit BASE_PATH if your tree differs)
# ---------------------------------------------------------------------------
CF_BASE_PATH    = "artifacts/implicite_use"  # not shipped — see DATA.md
CF_EXCEL_PATH   = "DATA/outputs/benchmark.csv"
CF_PROBA_DIR    = os.path.join(CF_BASE_PATH, "oof_proba_final")
CF_BEST_DIR     = os.path.join(CF_PROBA_DIR, "best_models")
CF_OUTPUT_DIR   = os.path.join(CF_PROBA_DIR, "confound_analysis")
os.makedirs(CF_OUTPUT_DIR, exist_ok=True)

CF_SEED = 42
np.random.seed(CF_SEED)

# Model files + thresholds (best thresholds from the nested CV grid search)
CF_MODELS = {
    "CamemBERT":    "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT":     "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LawMA":        "proba_oof_Lawma_BEST_cfg1_mean_last_MLP1.npy",
    "LLaMA":        "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL":         "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM":    "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet":     "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
CF_TFIDF_CSV = "tfidf_oof_probas.csv"
CF_BEST_ENSEMBLE_FILE = "proba_oof_BEST_ENSEMBLE_nested_cv.npy"

CF_THRESHOLDS = {
    "CamemBERT":    0.482666,
    "CamemBERTav2": 0.587369,
    "JuriBERT":     0.739345,
    "LawMA":        0.845000,
    "LLaMA":        0.614332,
    "SAUL":         0.793622,
    "ST-MiniLM":    0.539045,
    "ST-MPNet":     0.631491,
    "TF-IDF":       0.471811,
    "BEST_ENSEMBLE":0.610120,
}

# ---------------------------------------------------------------------------
# 2) Sanity check on Excel file
# ---------------------------------------------------------------------------
if not os.path.exists(CF_EXCEL_PATH):
    raise FileNotFoundError(
        f"Excel file not found at {CF_EXCEL_PATH}. "
        f"Edit CF_BASE_PATH at the top of the cell."
    )

print("=" * 80)
print("LOADING DATA")
print("=" * 80)

cf_df = pd.read_csv(CF_EXCEL_PATH)
cf_df = cf_df[["decision_id", "chunk_id", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]].copy()

# Resolve agree / disagree status (same logic as cell 33)
def _cf_extract(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
    if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
    return np.nan

cf_df["a"] = cf_df["eval_A1"].apply(_cf_extract)
cf_df["t"] = cf_df["eval_A2"].apply(_cf_extract)
cf_df["s"] = cf_df["eval_A3"].apply(_cf_extract)

def _cf_resolve(row):
    a, t, s = row["a"], row["t"], row["s"]
    if pd.notna(a) and pd.notna(t) and a == t: return a, "agree"
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s): return s, "disagree"
    return np.nan, "other"

_cf_res = cf_df.apply(_cf_resolve, axis=1)
cf_df["label_str"] = _cf_res.apply(lambda x: x[0])
cf_df["case"]      = _cf_res.apply(lambda x: x[1])
cf_df = cf_df[cf_df["label_str"].isin(["oui", "non"])].copy()
cf_df["label"] = (cf_df["label_str"] == "oui").astype(int)
cf_df = cf_df.reset_index(drop=True)

print(f"  n = {len(cf_df)}  |  agree = {(cf_df['case']=='agree').sum()}  |  "
      f"disagree = {(cf_df['case']=='disagree').sum()}")
print(f"  true negatives (label = 0): {(cf_df['label']==0).sum()}")

# ---------------------------------------------------------------------------
# 3) Compute confound features
# ---------------------------------------------------------------------------
print("\nComputing confound features...")

cf_df["chunk_len"]       = cf_df["text"].astype(str).str.split().str.len()
cf_df["article_len"]     = cf_df["article_text"].astype(str).str.split().str.len()
cf_df["log_chunk_len"]   = np.log(cf_df["chunk_len"].clip(lower=1))
cf_df["log_article_len"] = np.log(cf_df["article_len"].clip(lower=1))

# TF-IDF cosine similarity between chunk and article
# Vocabulary fitted on the benchmark itself (chunks ∪ articles)
_cf_vec = TfidfVectorizer(lowercase=True, strip_accents="unicode",
                          ngram_range=(1, 1), min_df=2)
_cf_texts = pd.concat([cf_df["text"].astype(str),
                       cf_df["article_text"].astype(str)], axis=0)
_cf_vec.fit(_cf_texts)
_cf_Xc = normalize(_cf_vec.transform(cf_df["text"].astype(str)),         axis=1)
_cf_Xa = normalize(_cf_vec.transform(cf_df["article_text"].astype(str)), axis=1)
cf_df["tfidf_overlap"] = np.asarray(_cf_Xc.multiply(_cf_Xa).sum(axis=1)).ravel()

print(cf_df[["chunk_len", "article_len", "tfidf_overlap"]].describe().round(3))

# Quick sanity: do confounds differ between agree/disagree on true negatives?
_cf_tn = cf_df[cf_df["label"] == 0]
print("\nConfound means by case (true negatives only):")
print(_cf_tn.groupby("case")[["chunk_len", "article_len", "tfidf_overlap"]]
            .mean().round(3).to_string())

# ---------------------------------------------------------------------------
# 4) Load OOF probabilities (skip missing files with a warning)
# ---------------------------------------------------------------------------
print("\nLoading OOF probabilities...")
cf_probas = {}

for name, fname in CF_MODELS.items():
    path = os.path.join(CF_PROBA_DIR, fname)
    if os.path.exists(path):
        cf_probas[name] = np.load(path)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} — file not found, skipping.")

_cf_tfidf_path = os.path.join(CF_PROBA_DIR, CF_TFIDF_CSV)
if os.path.exists(_cf_tfidf_path):
    cf_probas["TF-IDF"] = pd.read_csv(_cf_tfidf_path)["proba_oui"].values
    print("  ✓ TF-IDF")

_cf_ens_path = os.path.join(CF_BEST_DIR, CF_BEST_ENSEMBLE_FILE)
if os.path.exists(_cf_ens_path):
    cf_probas["BEST_ENSEMBLE"] = np.load(_cf_ens_path)
    print("  ✓ BEST_ENSEMBLE")
else:
    print(f"  ✗ BEST_ENSEMBLE not found at {_cf_ens_path}")

if len(cf_probas) == 0:
    raise RuntimeError("No model probabilities loaded. Check CF_PROBA_DIR.")

# Sanity: every proba vector must have the same length as cf_df
for _name, _p in cf_probas.items():
    if len(_p) != len(cf_df):
        raise ValueError(
            f"Length mismatch for {_name}: probas has {len(_p)} rows, "
            f"cf_df has {len(cf_df)}."
        )

# ---------------------------------------------------------------------------
# 5) Nested logistic regressions on FALSE POSITIVES (true negatives only)
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("NESTED LOGISTIC REGRESSIONS  is_FP ~ disagreement (+ confounds)")
print("Restricted to gold = NO  |  Cluster-robust SE on decision_id")
print("=" * 80)

CF_SPECS = {
    "M0_baseline": ["disagreement"],
    "M1_lengths":  ["disagreement", "log_chunk_len", "log_article_len"],
    "M2_full":     ["disagreement", "log_chunk_len", "log_article_len", "tfidf_overlap"],
}

cf_rows = []
for cf_model_name, cf_p_oui in cf_probas.items():
    cf_thr = CF_THRESHOLDS.get(cf_model_name, 0.5)

    cf_work = cf_df.copy()
    cf_work["y_pred"]       = (cf_p_oui >= cf_thr).astype(int)
    cf_work["is_FP"]        = ((cf_work["label"] == 0) & (cf_work["y_pred"] == 1)).astype(int)
    cf_work["disagreement"] = (cf_work["case"] == "disagree").astype(int)

    cf_tn = cf_work[cf_work["label"] == 0].copy()
    if cf_tn["is_FP"].nunique() < 2:
        print(f"  {cf_model_name}: degenerate (no FP variation), skipped.")
        continue

    for spec_name, covariates in CF_SPECS.items():
        X = sm.add_constant(cf_tn[covariates].astype(float))
        y = cf_tn["is_FP"].astype(int)
        clusters = cf_tn["decision_id"].astype(str).values

        try:
            mdl = sm.Logit(y, X).fit(
                cov_type="cluster",
                cov_kwds={"groups": clusters},
                disp=0, maxiter=200,
            )
            coef = mdl.params["disagreement"]
            se   = mdl.bse["disagreement"]
            pval = mdl.pvalues["disagreement"]
            OR   = np.exp(coef)
            ci_lo, ci_hi = np.exp(coef - 1.96 * se), np.exp(coef + 1.96 * se)
        except Exception as exc:
            print(f"  {cf_model_name} / {spec_name}: regression failed ({exc})")
            OR = ci_lo = ci_hi = pval = np.nan

        cf_rows.append({
            "model": cf_model_name, "spec": spec_name,
            "OR_disagree": OR, "CI_low": ci_lo, "CI_high": ci_hi,
            "p_disagree": pval,
            "n_FP": int(cf_tn["is_FP"].sum()),
            "n_TN": len(cf_tn),
        })

cf_res = pd.DataFrame(cf_rows)
cf_pivot_or = cf_res.pivot(index="model", columns="spec", values="OR_disagree").round(2)
cf_pivot_p  = cf_res.pivot(index="model", columns="spec", values="p_disagree").round(4)

print("\nOR for the disagreement coefficient (model × specification):")
print(cf_pivot_or.to_string())
print("\np-value for the disagreement coefficient:")
print(cf_pivot_p.to_string())

# Compact summary for the rebuttal
if "BEST_ENSEMBLE" in cf_pivot_or.index:
    print("\n--- Rebuttal-ready summary (BEST_ENSEMBLE) ---")
    for spec in CF_SPECS:
        r = cf_res[(cf_res["model"] == "BEST_ENSEMBLE") & (cf_res["spec"] == spec)].iloc[0]
        print(f"  {spec:14s}: OR = {r['OR_disagree']:.2f}  "
              f"95% CI [{r['CI_low']:.2f}, {r['CI_high']:.2f}]  p = {r['p_disagree']:.3f}")

cf_res.to_csv(os.path.join(CF_OUTPUT_DIR, "confound_regressions_long.csv"), index=False)
cf_pivot_or.to_csv(os.path.join(CF_OUTPUT_DIR, "confound_OR_pivot.csv"))
cf_pivot_p.to_csv(os.path.join(CF_OUTPUT_DIR, "confound_p_pivot.csv"))
print(f"\n✓ CSVs saved to {CF_OUTPUT_DIR}")

# ---------------------------------------------------------------------------
# 6) Stratified robustness check (BEST_ENSEMBLE)
# ---------------------------------------------------------------------------
if "BEST_ENSEMBLE" in cf_probas:
    print("\n" + "=" * 80)
    print("STRATIFIED FPR (BEST_ENSEMBLE) — robustness without regression")
    print("=" * 80)

    cf_work = cf_df.copy()
    cf_work["y_pred"] = (cf_probas["BEST_ENSEMBLE"] >= CF_THRESHOLDS["BEST_ENSEMBLE"]).astype(int)
    cf_work["is_FP"]  = ((cf_work["label"] == 0) & (cf_work["y_pred"] == 1)).astype(int)
    cf_tn = cf_work[cf_work["label"] == 0].copy()

    for confound, label in [("tfidf_overlap", "TF-IDF overlap"),
                            ("chunk_len",     "Chunk length"),
                            ("article_len",   "Article length")]:
        cf_tn["q"] = pd.qcut(cf_tn[confound], q=4,
                             labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
        strat = cf_tn.groupby(["q", "case"], observed=True).agg(
            n=("is_FP", "size"), FPR=("is_FP", "mean"),
        ).round(3).reset_index()
        wide = strat.pivot(index="q", columns="case", values="FPR")
        if "agree" in wide.columns and "disagree" in wide.columns:
            wide["gap_pp"] = (wide["disagree"] - wide["agree"]) * 100
        print(f"\n{label} quartiles  (FPR per group, gap in pp):")
        print(wide.round(3).to_string())
        wide.to_csv(os.path.join(CF_OUTPUT_DIR, f"stratified_FPR_{confound}.csv"))

# ---------------------------------------------------------------------------
# 7) Forest plot — OR(disagreement) under each spec, all models
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
cf_offsets = {"M0_baseline": -0.22, "M1_lengths": 0.0, "M2_full": +0.22}
cf_colors  = {"M0_baseline": "#3498db", "M1_lengths": "#f39c12", "M2_full": "#e74c3c"}
cf_models_order = [m for m in cf_pivot_or.index if m != "BEST_ENSEMBLE"]
if "BEST_ENSEMBLE" in cf_pivot_or.index:
    cf_models_order.append("BEST_ENSEMBLE")
cf_y_pos = {m: i for i, m in enumerate(cf_models_order)}

_seen_specs = set()
for _, r in cf_res.iterrows():
    if pd.isna(r["OR_disagree"]):
        continue
    y = cf_y_pos[r["model"]] + cf_offsets[r["spec"]]
    label = r["spec"] if r["spec"] not in _seen_specs else None
    _seen_specs.add(r["spec"])
    ax.errorbar(r["OR_disagree"], y,
                xerr=[[r["OR_disagree"] - r["CI_low"]],
                      [r["CI_high"] - r["OR_disagree"]]],
                fmt="o", capsize=3, markersize=6,
                color=cf_colors[r["spec"]],
                ecolor=cf_colors[r["spec"]],
                label=label)

ax.axvline(1.0, ls="--", c="gray", alpha=0.6)
ax.set_yticks(list(cf_y_pos.values()))
ax.set_yticklabels(list(cf_y_pos.keys()))
ax.set_xlabel("Odds ratio for disagreement coefficient (95% CI)")
ax.set_title("Disagreement → FP: robustness to surface confounds")
ax.legend(title="Specification", loc="lower right")
plt.tight_layout()
fig.savefig(os.path.join(CF_OUTPUT_DIR, "fig_confound_forest.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(CF_OUTPUT_DIR, "fig_confound_forest.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n" + "=" * 80)
print(f"DONE — outputs in {CF_OUTPUT_DIR}")
print("=" * 80)